## 05.04节练习参考答案

### 环境准备

In [ ]:
%pip install pypto==0.2.0 torch torch_npu numpy

In [ ]:
import os, sys
os.environ["TILE_FWK_DEVICE_ID"] = "0"

# 本 notebook 位于 answers/ 子目录下，src 包在其上一级目录。
# 把上层目录加入 sys.path，才能 from src.pto_layers import ...
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import pypto
import torch
import torch_npu
import torch.nn as nn
import torch.fft as fft
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import math

from src.pto_layers import PyPTOLinear, PyPTOReLU, PyPTOLazyLinear

本节的解答思路参考了 [《动手学深度学习》习题解答](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/ch05/ch05) ，并在此基础上补充了 PyPTO 的实现。  

### 练习5.4.1
设计一个接受输入并计算张量降维的层，它返回$$y_k = \sum_{i,j} W_{ijk} x_i x_j$$

**解答：**
这个公式表示一个线性变换，将输入张量$x$中所有可能的二元组$(x_i, x_j)$进行组合，并对它们进行加权求和。其中，$W_{ijk}$表示权重张量中第$i,j,k$个元素的值。具体而言，该公式计算了输入张量$x$中所有二元组$(x_i, x_j)$对应的特征向量$u_{ij}$为：
$$u_{ij} = x_i \cdot x_j$$
然后，根据权重张量$W$中的权重$W_{ijk}$，对所有特征向量$u_{ij}$进行线性组合，得到输出向量$y_k$为：
$$y_k = \sum_{i,j} W_{ijk} u_{ij} = \sum_{i,j} W_{ijk} x_i x_j$$
该操作可以被视为一种降维操作，将高维输入$x$映射到低维输出空间$y$中。

In [2]:
class TensorReduction(nn.Module):
    def __init__(self, dim1, dim2):
        super(TensorReduction, self).__init__()
        # 定义一个可训练的权重参数，维度为(dim2, dim1, dim1)
        self.weight = nn.Parameter(torch.rand(dim2, dim1, dim1))

    def forward(self, X):
        # 初始化一个全零张量，大小为(X.shape[0], self.weight.shape[0])
        Y = torch.zeros(X.shape[0], self.weight.shape[0])
        for k in range(self.weight.shape[0]):
            # 计算temp = X @ weight[k] @ X^T
            temp = torch.matmul(X, self.weight[k]) @ X.T
            # 取temp的对角线元素，存入Y[:, k]
            Y[:, k] = temp.diagonal()
        return Y

# 创建一个TensorReduction层，dim1=10, dim2=5
layer = TensorReduction(10, 5)
# 创建一个大小为(2, 10)的张量X
X = torch.rand(2, 10)
# 对layer(X)进行前向传播，返回一个大小为(2, 5)的张量
layer(X).shape

torch.Size([2, 5])

**PyPTO 版**

In [3]:
class TensorReduction(nn.Module):
    def __init__(self, dim1, dim2):
        super(TensorReduction, self).__init__()
        # 定义一个可训练的权重参数，维度为(dim2, dim1, dim1)
        self.weight = nn.Parameter(torch.rand(dim2, dim1, dim1).npu())

    def forward(self, X):
        # 初始化一个全零张量，大小为(X.shape[0], self.weight.shape[0])
        Y = torch.zeros(X.shape[0], self.weight.shape[0]).npu()
        for k in range(self.weight.shape[0]):
            # 计算temp = X @ weight[k] @ X^T
            temp = torch.matmul(X, self.weight[k]) @ X.T
            # 取temp的对角线元素，存入Y[:, k]
            Y[:, k] = temp.diagonal()
        return Y

# 创建一个TensorReduction层，dim1=10, dim2=5
layer = TensorReduction(10, 5)
# 创建一个大小为(2, 10)的张量X
X = torch.rand(2, 10).npu()
# 对layer(X)进行前向传播，返回一个大小为(2, 5)的张量
layer(X).shape

torch.Size([2, 5])

### 练习5.4.2
设计一个返回输入数据的傅立叶系数前半部分的层。

**解答：**
根据维基百科中：

傅里叶级数是将任意周期函数表示为一组正弦和余弦函数的无限级数的方法。假设$f(x)$是在区间$[-L,L]$中定义的一个函数，其周期为$2L$，则其傅里叶级数可表示为：

$$
f(x) = \frac{a_0}{2} + \sum_{n=1}^{\infty}\left[a_n \cos\left(\frac{n\pi x}{L}\right) + b_n \sin\left(\frac{n\pi x}{L}\right)\right]
$$

其中，系数$a_0,a_n$和$b_n$可以通过以下公式计算得出：

$$
a_0 = \frac{1}{2L}\int_{-L}^{L} f(x)dx
$$

$$
a_n = \frac{1}{L}\int_{-L}^{L} f(x)\cos\left(\frac{n\pi x}{L}\right)dx,\; n > 0
$$

$$
b_n = \frac{1}{L}\int_{-L}^{L} f(x)\sin\left(\frac{n\pi x}{L}\right)dx,\; n > 0
$$

系数$a_n$和$b_n$实际上是$f(x)$与$\cos\left(\frac{n\pi x}{L}\right)$和$\sin\left(\frac{n\pi x}{L}\right)$的内积，而$a_0$是$f(x)$平均值的一半。

在torch中有相应的函数可以轻松的实现傅里叶级数，如下代码所示：

In [4]:
class FourierLayer(nn.Module):
    def __init__(self):
        super(FourierLayer, self).__init__()

    def forward(self, x):
        # 对输入的张量 x 进行快速傅里叶变换
        x = fft.fftn(x)
        # 取出第三个维度的前半部分，即去掉直流分量和镜像分量
        x = x[:, :, :x.shape[2] // 2]
        # 返回处理后的张量
        return x

# 创建一个随机数值为 [0, 1) 的形状为 (1, 2, 5) 的张量 X
X = torch.rand(1, 2, 5)
# 实例化一个 FourierLayer 的网络对象 net
net = FourierLayer()
# 将 X 输入到网络 net 中进行前向计算，并输出结果
net(X)

tensor([[[ 5.1658+0.0000j, -0.5036-0.1989j],
         [ 0.5550+0.0000j, -0.2381+1.1614j]]])

**PyPTO 版**

In [6]:
class FourierLayer(nn.Module):
    def __init__(self):
        super(FourierLayer, self).__init__()

    def forward(self, x):
        # 对输入的张量 x 进行快速傅里叶变换
        x = fft.fftn(x)
        # 取出第三个维度的前半部分，即去掉直流分量和镜像分量
        x = x[:, :, :x.shape[2] // 2]
        # 返回处理后的张量
        return x

# 创建一个随机数值为 [0, 1) 的形状为 (1, 2, 5) 的张量 X
X = torch.rand(1, 2, 5).npu()
# 实例化一个 FourierLayer 的网络对象 net
net = FourierLayer()
# 将 X 输入到网络 net 中进行前向计算，并输出结果
net(X)

tensor([[[4.0679+0.0000e+00j, 0.1323-6.7176e-01j],
         [0.8749-1.9551e-16j, 0.4997-6.3180e-01j]]], device='npu:0')